In [1]:
2#  !pip install requests pandas tqdm

import os
import re
import time
import json
import csv
import requests
import pandas as pd
from tqdm import tqdm
from typing import List, Dict, Optional

# ---------------------------
# CONFIG - Replace API_KEY
# ---------------------------
API_KEY = "JhnIenwb3246rhRB3pNlw5FR2MuiNx5M4BCQ9DhS"  # <--Semantic Scholar API key
DATA_DIR = "data"
PDF_DIR = os.path.join(DATA_DIR, "pdfs")
METADATA_JSON = os.path.join(DATA_DIR, "metadata.json")
METADATA_CSV = os.path.join(DATA_DIR, "metadata.csv")
SEARCH_LIMIT = 12  # number of results to fetch by default

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ---------------------------
# SMART SUGGESTIONS (Feature 1)
# ---------------------------
SUGGESTIONS = {
    "ai": ["artificial intelligence healthcare", "deep learning medical imaging", "explainable AI"],
    "nlp": ["transformers for text summarization", "nlp in healthcare"],
    "ml": ["machine learning for diagnosis", "ml anomaly detection"],
    "covid": ["covid-19 vaccine efficacy", "covid-19 transmission modeling"]
}

def auto_suggest(topic: str):
    key = topic.lower().strip()
    printed = False
    for short, suggestions in SUGGESTIONS.items():
        if key == short:
            print("\n💡 Suggested research keyword variations:")
            for s in suggestions:
                print("   🔸", s)
            printed = True
    if not printed and len(topic.split()) <= 2:
        # show general hints
        print("\n💡 Tip: Consider adding domain or method words (e.g., 'healthcare', 'CNN', 'transformer') for better search results.")
    print()

# ---------------------------
# UTILITIES
# ---------------------------
def sanitize_filename(name: str, max_len: int = 100) -> str:
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name[:max_len].strip()

KEYWORD_TAGS = ["disease", "covid", "classification", "deep", "cnn", "transformer",
                "diagnosis", "health", "medical", "image", "nlp", "survey", "review", "anomaly"]

def auto_tags(title: str) -> List[str]:
    title_lower = title.lower()
    tags = [kw for kw in KEYWORD_TAGS if kw in title_lower]
    # also add words longer than 5 chars that appear frequently as naive tag (not too noisy)
    extra = [w for w in set(re.findall(r'\b[a-z]{6,}\b', title_lower)) if w not in tags][:3]
    return list(dict.fromkeys(tags + extra))  # preserve order, uniq

def title_relevance_score(title: str, query: str) -> int:
    # simple scoring: keyword exact matches + token overlap
    title_l = title.lower()
    query_tokens = [t for t in re.findall(r'\w+', query.lower()) if len(t) > 1]
    score = 0
    for t in query_tokens:
        if t in title_l:
            score += 2
        # partial token in title: +1
        for word in re.findall(r'\w+', title_l):
            if t in word and t != word:
                score += 1
    return score

def is_duplicate(title: str, existing_meta: List[Dict]) -> bool:
    t = title.lower().strip()
    for m in existing_meta:
        if m.get("title","").lower().strip() == t:
            return True
    return False

def load_existing_metadata() -> List[Dict]:
    if os.path.exists(METADATA_JSON):
        try:
            with open(METADATA_JSON, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return []
    return []

def save_metadata_list(metadata: List[Dict]):
    with open(METADATA_JSON, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    if metadata:
        keys = metadata[0].keys()
        with open(METADATA_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(keys))
            writer.writeheader()
            writer.writerows(metadata)

# ---------------------------
# SEMANTIC SCHOLAR SEARCH (core)
# ---------------------------
def semantic_scholar_search(query: str, limit: int = SEARCH_LIMIT) -> List[Dict]:
    base = "https://api.semanticscholar.org/graph/v1/paper/search"
    fields = "title,authors,year,url,isOpenAccess,openAccessPdf,externalIds,abstract"
    params = {"query": query, "limit": limit, "fields": fields}
    headers = {"x-api-key": API_KEY}
    resp = requests.get(base, params=params, headers=headers, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    papers = []
    for item in data.get("data", []):
        title = item.get("title", "No title")
        authors = ", ".join([a.get("name","") for a in item.get("authors", [])]) if item.get("authors") else ""
        year = item.get("year", None)
        url = item.get("url", None)
        is_open = bool(item.get("isOpenAccess", False))
        pdf_url = None
        if item.get("openAccessPdf"):
            pdf_url = item["openAccessPdf"].get("url")
        abstract = item.get("abstract", "")
        external = item.get("externalIds", {})
        papers.append({
            "title": title,
            "authors": authors,
            "year": year,
            "url": url,
            "is_open_access": is_open,
            "pdf_url": pdf_url,
            "external_ids": external,
            "abstract": abstract
        })
    return papers

# ---------------------------
# DISPLAY / INSIGHTS (Feature 2 & 3)
# ---------------------------
def prepare_and_rank(papers: List[Dict], query: str, existing_meta: List[Dict]) -> List[Dict]:
    for p in papers:
        p["score"] = title_relevance_score(p["title"], query)
        p["tags"] = auto_tags(p["title"])
        p["is_duplicate"] = is_duplicate(p["title"], existing_meta)
    # sort by score (desc) then year (desc if exists)
    papers_sorted = sorted(papers, key=lambda x: (x["score"], x.get("year") or 0), reverse=True)
    return papers_sorted

def emoji_access(p):
    return "🔓" if p["is_open_access"] else "🔒"

def emoji_pdf(p):
    return "📄" if p.get("pdf_url") else "❌"

def show_insights_table(papers: List[Dict]):
    rows = []
    for i,p in enumerate(papers):
        rows.append({
            "Index": i,
            "Title (short)": (p["title"][:70] + "...") if len(p["title"])>70 else p["title"],
            "Year": p.get("year"),
            "Authors": (p["authors"][:40] + "...") if p.get("authors") and len(p["authors"])>40 else p.get("authors"),
            "Access": emoji_access(p),
            "PDF": emoji_pdf(p),
            "Score": p.get("score", 0),
            "Tags": ", ".join(p.get("tags", [])),
            "Duplicate": "⚠" if p.get("is_duplicate") else ""
        })
    df = pd.DataFrame(rows)
    display(df)
    return df

# ---------------------------
# DOWNLOAD LOGIC
# ---------------------------
def download_file(url: str, dest_path: str, max_retries: int = 3, retry_delay: float = 1.2) -> bool:
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, stream=True, timeout=30)
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            with open(dest_path, "wb") as f:
                if total and total > 0:
                    pbar = tqdm(total=total, unit="B", unit_scale=True, desc=f"Downloading", leave=False)
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
                    pbar.close()
                else:
                    f.write(r.content)
            return True
        except Exception as e:
            print(f"Warning: attempt {attempt} failed for URL: {url[:80]}... Error: {e}")
            time.sleep(retry_delay * attempt)
    return False

def download_selected_papers(papers: List[Dict], selected_indices: List[int], existing_meta: List[Dict]) -> List[Dict]:
    metadata_entries = []
    for idx in selected_indices:
        if idx < 0 or idx >= len(papers):
            print(f"Index {idx} out of range — skipped.")
            continue
        p = papers[idx]
        title_clean = sanitize_filename(p["title"]) or f"paper_{idx}"
        filename = f"{p.get('year') or 'na'}_{title_clean}.pdf"
        dest_path = os.path.join(PDF_DIR, filename)

        entry = {
            "index_selected": idx,
            "title": p["title"],
            "authors": p.get("authors"),
            "year": p.get("year"),
            "source_url": p.get("url"),
            "pdf_url": p.get("pdf_url"),
            "local_path": None,
            "status": None,
            "score": p.get("score"),
            "tags": p.get("tags"),
            "is_duplicate": p.get("is_duplicate")
        }

        if p.get("is_duplicate"):
            entry["status"] = "duplicate_skipped"
            print(f"⚠ Duplicate detected - skipped: {p['title']}")
            metadata_entries.append(entry)
            continue

        if not p.get("pdf_url"):
            entry["status"] = "skipped_no_pdf"
            print(f"⚠ Skipped (no open PDF): {p['title']}")
            metadata_entries.append(entry)
            continue

        print(f"📥 Downloading [{idx}] {p['title']}")
        ok = download_file(p["pdf_url"], dest_path)
        if ok:
            entry["local_path"] = dest_path
            entry["status"] = "downloaded"
            print(f"✅ Saved to {dest_path}")
        else:
            entry["status"] = "failed_download"
            print(f"❌ Failed to download: {p['title']}")
        metadata_entries.append(entry)
    return metadata_entries

# ---------------------------
# MAIN INTERACTIVE FLOW
# ---------------------------
def run_enhanced_flow():
    print("=== Enhanced Milestone-1: Paper Retriever (Your Independent Version) ===\n")
    existing_meta = load_existing_metadata()

    topic = input("Enter your research topic / query: ").strip()
    if not topic:
        print("Topic empty. Exiting.")
        return

    # Feature 1: Smart suggestions
    auto_suggest(topic)

    # Ask year filter optionally (user chose B earlier)
    apply_filter = input("Do you want to filter results by year range? (y/n): ").strip().lower().startswith("y")
    start_year = end_year = None
    if apply_filter:
        try:
            start_year = int(input("Enter start year (e.g., 2019): ").strip())
            end_year = int(input("Enter end year (e.g., 2025): ").strip())
            if start_year > end_year:
                start_year, end_year = end_year, start_year
        except Exception:
            print("Invalid input for years; continuing without year filter.")
            apply_filter = False

    # Perform search
    print("\n🔎 Searching Semantic Scholar...")
    papers = semantic_scholar_search(topic, limit=SEARCH_LIMIT)
    if not papers:
        print("No papers found.")
        return

    # Feature 2 & 4 & 5: ranking, tags, duplicate detection
    papers = prepare_and_rank(papers, topic, existing_meta)

    # optional year filtering
    if apply_filter:
        papers_filtered = [p for p in papers if p.get("year") and start_year <= int(p["year"]) <= end_year]
        print(f"Filtered: {len(papers_filtered)} out of {len(papers)} match year range {start_year}-{end_year}.")
        papers = papers_filtered
        if not papers:
            print("No papers after filtering by year.")
            return

    # show insights table (feature 3)
    print("\n📊 Results insights:")
    show_insights_table(papers)

    # selection
    selection_str = input("Select paper indices to download (comma-separated, e.g. 0,2,3) or 'all' to attempt all: ").strip()
    if selection_str.lower() == "all":
        selected_indices = list(range(len(papers)))
    else:
        try:
            selected_indices = [int(s.strip()) for s in selection_str.split(",") if s.strip()!=""]
        except Exception:
            print("Invalid selection. Exiting.")
            return

    print(f"Selected indices: {selected_indices}")
    if not input("Proceed with downloads? (y/n): ").strip().lower().startswith("y"):
        print("Cancelled.")
        return

    # download selected
    metadata_new = download_selected_papers(papers, selected_indices, existing_meta)

    # combine metadata (append)
    combined_meta = existing_meta + metadata_new
    save_metadata_list(combined_meta)
    print(f"\n📁 Metadata saved to: {METADATA_JSON} and {METADATA_CSV}")
    print(f"📂 PDFs saved under: {PDF_DIR}")

    # show summary table
    if metadata_new:
        display(pd.DataFrame(metadata_new))
    else:
        print("No new metadata generated.")

# Run it:
if __name__ == "__main__":
    run_enhanced_flow()


=== Enhanced Milestone-1: Paper Retriever (Your Independent Version) ===

Enter your research topic / query: healthcare

💡 Tip: Consider adding domain or method words (e.g., 'healthcare', 'CNN', 'transformer') for better search results.

Do you want to filter results by year range? (y/n): y
Enter start year (e.g., 2019): 2019
Enter end year (e.g., 2025): 2025

🔎 Searching Semantic Scholar...
Filtered: 9 out of 12 match year range 2019-2025.

📊 Results insights:


,Index,Title (short),Year,Authors,Access,PDF,Score,Tags,Duplicate
0,0,A Review of Large Language Models in Medical E...,2025,"J. Vrdoljak, Zvonimir Boban, Marino Vilo...",🔓,📄,2,"health, medical, review, healthcare, models, d...",⚠
1,1,Understanding Psychosocial Barriers to Healthc...,2025,"Ann Thong Lee, R. Ramasamy, Anusuyah Sub...",🔓,📄,2,"health, review, psychosocial, barriers, unders...",⚠
2,2,Transformative Potential of AI in Healthcare: ...,2024,"Molly Bekbolatova, Jonathan Mayer, Chiwe...",🔓,📄,2,"health, ethical, definitions, navigating",⚠
3,3,Global Regulatory Frameworks for the Use of Ar...,2024,"K. Palaniappan, Elaine Yan Ting Lin, Sil...",🔓,📄,2,"health, sector, artificial, global",⚠
4,4,The Role of AI in Hospitals and Clinics: Trans...,2024,"Shiva Maleki Varnosfaderani, Mohamad For...",🔓,📄,2,"health, hospitals, century, clinics",⚠
5,5,"ChatGPT Utility in Healthcare Education, Resea...",2023,Malik Sallam,🔓,📄,2,"health, review, research, concerns, education",⚠
6,6,Revolutionizing healthcare: the role of artifi...,2023,"Shuroug A. Alowais, Sahar S. Alghamdi, N...",🔓,📄,2,"health, artificial, clinical, practice",⚠
7,7,A guide to deep learning in healthcare,2019,"A. Esteva, Alexandre Robicquet, Bharath ...",🔒,❌,2,"deep, health, healthcare, learning",⚠
8,8,The potential for artificial intelligence in h...,2019,"T. Davenport, R. Kalakota",🔓,📄,2,"health, healthcare, artificial, potential",⚠


Select paper indices to download (comma-separated, e.g. 0,2,3) or 'all' to attempt all: all
Selected indices: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Proceed with downloads? (y/n): y
⚠ Duplicate detected - skipped: A Review of Large Language Models in Medical Education, Clinical Decision Support, and Healthcare Administration
⚠ Duplicate detected - skipped: Understanding Psychosocial Barriers to Healthcare Technology Adoption: A Review of TAM Technology Acceptance Model and Unified Theory of Acceptance and Use of Technology and UTAUT Frameworks
⚠ Duplicate detected - skipped: Transformative Potential of AI in Healthcare: Definitions, Applications, and Navigating the Ethical Landscape and Public Perspectives
⚠ Duplicate detected - skipped: Global Regulatory Frameworks for the Use of Artificial Intelligence (AI) in the Healthcare Services Sector
⚠ Duplicate detected - skipped: The Role of AI in Hospitals and Clinics: Transforming Healthcare in the 21st Century
⚠ Duplicate detected - skipped: ChatGPT

,index_selected,title,authors,year,source_url,pdf_url,local_path,status,score,tags,is_duplicate
0,0,A Review of Large Language Models in Medical E...,"J. Vrdoljak, Zvonimir Boban, Marino Vilović, M...",2025,https://www.semanticscholar.org/paper/0c561f33...,https://doi.org/10.3390/healthcare13060603,None,duplicate_skipped,2,"[health, medical, review, healthcare, models, ...",True
1,1,Understanding Psychosocial Barriers to Healthc...,"Ann Thong Lee, R. Ramasamy, Anusuyah Subbarao",2025,https://www.semanticscholar.org/paper/aeffb4aa...,https://doi.org/10.3390/healthcare13030250,None,duplicate_skipped,2,"[health, review, psychosocial, barriers, under...",True
2,2,Transformative Potential of AI in Healthcare: ...,"Molly Bekbolatova, Jonathan Mayer, Chiwei Ong,...",2024,https://www.semanticscholar.org/paper/467411fd...,https://www.mdpi.com/2227-9032/12/2/125/pdf?ve...,None,duplicate_skipped,2,"[health, ethical, definitions, navigating]",True
3,3,Global Regulatory Frameworks for the Use of Ar...,"K. Palaniappan, Elaine Yan Ting Lin, Silke Vogel",2024,https://www.semanticscholar.org/paper/2fba8788...,https://www.mdpi.com/2227-9032/12/5/562/pdf?ve...,None,duplicate_skipped,2,"[health, sector, artificial, global]",True
4,4,The Role of AI in Hospitals and Clinics: Trans...,"Shiva Maleki Varnosfaderani, Mohamad Forouzanfar",2024,https://www.semanticscholar.org/paper/a779f5dd...,https://www.mdpi.com/2306-5354/11/4/337/pdf?ve...,None,duplicate_skipped,2,"[health, hospitals, century, clinics]",True
5,5,"ChatGPT Utility in Healthcare Education, Resea...",Malik Sallam,2023,https://www.semanticscholar.org/paper/dfdf7ff0...,https://www.mdpi.com/2227-9032/11/6/887/pdf?ve...,None,duplicate_skipped,2,"[health, review, research, concerns, education]",True
6,6,Revolutionizing healthcare: the role of artifi...,"Shuroug A. Alowais, Sahar S. Alghamdi, Nada Al...",2023,https://www.semanticscholar.org/paper/5cde4748...,https://bmcmededuc.biomedcentral.com/counter/p...,None,duplicate_skipped,2,"[health, artificial, clinical, practice]",True
7,7,A guide to deep learning in healthcare,"A. Esteva, Alexandre Robicquet, Bharath Ramsun...",2019,https://www.semanticscholar.org/paper/44e1dd74...,,None,duplicate_skipped,2,"[deep, health, healthcare, learning]",True
8,8,The potential for artificial intelligence in h...,"T. Davenport, R. Kalakota",2019,https://www.semanticscholar.org/paper/ddf4172c...,https://www.rcpjournals.org/content/futurehosp...,None,duplicate_skipped,2,"[health, healthcare, artificial, potential]",True


In [2]:
!pip install pymupdf nltk scikit-learn


In [3]:
import os

PDF_DIR = "data/pdfs"

print("Folder exists:", os.path.exists(PDF_DIR))

if os.path.exists(PDF_DIR):
    files = os.listdir(PDF_DIR)
    print("Files inside data/pdfs:", files)
else:
    print("❌ data/pdfs folder not found")


Folder exists: True
Files inside data/pdfs: ['2023_Revolutionizing healthcare the role of artificial intelligence in clinical practice.pdf']


In [4]:
## Extract text
import fitz

files = os.listdir(PDF_DIR)

if not files:
    print("❌ No PDFs found. Upload or download PDFs first.")
else:
    pdf_path = os.path.join(PDF_DIR, files[0])
    print("Using PDF:", pdf_path)

    doc = fitz.open(pdf_path)
    print("Total pages:", len(doc))

    text = ""
    for page in doc:
        text += page.get_text()

    print("\n🔹 STEP 2 OUTPUT (first 500 chars):\n")
    print(text[:500])
    print("\nTotal characters extracted:", len(text))


Using PDF: data/pdfs/2023_Revolutionizing healthcare the role of artificial intelligence in clinical practice.pdf
Total pages: 15

🔹 STEP 2 OUTPUT (first 500 chars):

REVIEW
Open Access
© The Author(s) 2023. Open Access  This article is licensed under a Creative Commons Attribution 4.0 International License, which permits use, 
sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the original author(s) and 
the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or other third party material in this 
article are included in the article’s Creative

Total characters extracted: 86776


In [5]:
## Split into sections
import re

def split_into_sections(text):
    patterns = {
        "abstract": r"\babstract\b",
        "introduction": r"\bintroduction\b",
        "methodology": r"\b(method|methodology)\b",
        "results": r"\bresults\b",
        "conclusion": r"\b(conclusion|discussion)\b"
    }

    text_l = text.lower()
    indices = {k: re.search(v, text_l).start()
               for k, v in patterns.items() if re.search(v, text_l)}

    sections = {}
    sorted_items = sorted(indices.items(), key=lambda x: x[1])

    for i, (sec, start) in enumerate(sorted_items):
        end = sorted_items[i+1][1] if i+1 < len(sorted_items) else len(text)
        sections[sec] = text[start:end]

    return sections


sections = split_into_sections(text)

print("\n🔹 STEP 3 OUTPUT – Sections Found:")
print(sections.keys())



🔹 STEP 3 OUTPUT – Sections Found:
dict_keys(['abstract', 'introduction', 'results', 'conclusion', 'methodology'])


In [6]:
# Key findings
import nltk

# Ensure necessary NLTK resources are available
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize

# Function to extract key findings (sentence-wise)
def extract_key_findings(text):
    if not text:
        return []
    # Split text into sentences
    sentences = sent_tokenize(text)
    # Here you can add further filtering or keyword extraction if needed
    return sentences

# Example usage with sections dictionary
# sections = {'Introduction': 'Your text here...', 'Results': 'Your results text...'}
for sec, content in sections.items():
    print(f"\n{sec.upper()}:")
    for s in extract_key_findings(content):
        print("•", s)



ABSTRACT:
• Abstract

INTRODUCTION:
• Introduction  Healthcare systems are complex and challenging for all stakeholders, but artificial intelligence (AI) has 
transformed various fields, including healthcare, with the potential to improve patient care and quality of life.
• Rapid 
AI advancements can revolutionize healthcare by integrating it into clinical practice.
• Reporting AI’s role in clinical 
practice is crucial for successful implementation by equipping healthcare providers with essential knowledge and 
tools.
• Research Significance  This review article provides a comprehensive and up-to-date overview of the current state 
of AI in clinical practice, including its potential applications in disease diagnosis, treatment recommendations, and 
patient engagement.
• It also discusses the associated challenges, covering ethical and legal considerations and the 
need for human expertise.
• By doing so, it enhances understanding of AI’s significance in healthcare and supports 
healt

In [9]:
# -------------------------------------------------------
# REQUIRED FUNCTION: PDF TEXT EXTRACTION
# -------------------------------------------------------

import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    """
    Extracts text from a single PDF file
    """
    text = ""
    doc = fitz.open(pdf_path)
    for page in doc:
        text += page.get_text()
    return text


In [16]:
import re

def split_into_sections(text):
    """
    Splits text into common research sections
    """
    patterns = {
        "abstract": r"\babstract\b",
        "introduction": r"\bintroduction\b",
        "methodology": r"\b(method|methodology)\b",
        "results": r"\bresults\b",
        "conclusion": r"\b(conclusion|discussion)\b"
    }

    text_l = text.lower()
    indices = {k: re.search(v, text_l).start()
               for k, v in patterns.items() if re.search(v, text_l)}

    sections = {}
    sorted_items = sorted(indices.items(), key=lambda x: x[1])

    for i, (sec, start) in enumerate(sorted_items):
        end = sorted_items[i+1][1] if i+1 < len(sorted_items) else len(text)
        sections[sec] = text[start:end]

    return sections


In [17]:
# Process all PDFs
print("\n🔹 STEP 5 OUTPUT – All PDFs")

for pdf in os.listdir(PDF_DIR):
    path = os.path.join(PDF_DIR, pdf)
    text = extract_text_from_pdf(path)
    sections = split_into_sections(text)

    print("\nPDF:", pdf)
    print("Sections:", list(sections.keys()))




🔹 STEP 5 OUTPUT – All PDFs

PDF: 2023_Revolutionizing healthcare the role of artificial intelligence in clinical practice.pdf
Sections: ['abstract', 'introduction', 'results', 'conclusion', 'methodology']


In [18]:
# -------------------------------------------------------
# WEEK 5–6: OFFLINE DRAFT GENERATION + APA REFERENCES
# - Generates Abstract, Methods, Results
# - Synthesizes multiple papers
# - Formats APA references from metadata.csv
# - No OpenAI API required
# -------------------------------------------------------


import os
import pandas as pd

OUTPUT_DIR = "drafts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_DIR = "data"  # folder containing metadata.csv
METADATA_CSV = os.path.join(DATA_DIR, "metadata.csv")

def generate_apa_references(metadata_file, output_file):
    if not os.path.exists(metadata_file):
        print(f"❌ Metadata file '{metadata_file}' not found. Skipping references.")
        return
    df = pd.read_csv(metadata_file)
    references = []
    for _, row in df.iterrows():
        authors = row.get('authors', 'Unknown Author')
        year = row.get('year', 'n.d.')
        title = row.get('title', 'Untitled')
        venue = row.get('venue', 'Unknown Journal')
        references.append(f"{authors} ({year}). {title}. {venue}.")
    with open(os.path.join(OUTPUT_DIR, output_file), "w", encoding="utf-8") as f:
        f.write("\n\n".join(references))
    print(f"✅ APA references saved to {output_file}")

# Generate references
generate_apa_references(METADATA_CSV, "references.txt")


✅ APA references saved to references.txt


In [22]:
# -------------------------------------------------------
# WEEK 7: OFFLINE REVIEW & REFINEMENT
# Adds simple headers and formatting to Week 5–6 drafts
# No GPT / API required
# -------------------------------------------------------

import os
DRAFT_DIR = "/content/drafts"
REVISED_DIR = os.path.join(DRAFT_DIR, "revised")
os.makedirs(REVISED_DIR, exist_ok=True)


# Draft files from Week 5–6
DRAFT_FILES = ["abstract.txt", "methods.txt", "results.txt"]

def refine_text_offline(text, section_name):
    """
    Simple refinement:
    - Adds section header
    - Strips extra spaces
    - Can be extended to bullet points or highlights
    """
    return f"### {section_name.upper()} ###\n\n{text.strip()}"

# Loop through each draft and create refined version
for file in DRAFT_FILES:
    path = os.path.join(OUTPUT_DIR, file)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        section_name = file.replace(".txt", "")
        refined_text = refine_text_offline(text, section_name)
        with open(os.path.join(REVISED_DIR, file), "w", encoding="utf-8") as f:
            f.write(refined_text)
        print(f"✅ Refined {file}")
    else:
        print(f"⚠️ File {file} not found, skipping.")


✅ Refined abstract.txt
✅ Refined methods.txt
✅ Refined results.txt


In [23]:
# -------------------------------------------------------
# WEEK 8: OFFLINE GRADIO UI
# Displays Abstract, Methods, Results, and References
# Completely offline (no API key needed)
# -------------------------------------------------------

import os
import gradio as gr

OUTPUT_DIR = "drafts"
REVISED_DIR = os.path.join(OUTPUT_DIR, "revised")

def load_file(path):
    """
    Loads text from a file or returns a placeholder if missing
    """
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    return "File not found."

def show_final_content():
    """
    Returns all finalized sections for the UI
    """
    return (
        load_file(os.path.join(REVISED_DIR, "abstract.txt")),
        load_file(os.path.join(REVISED_DIR, "methods.txt")),
        load_file(os.path.join(REVISED_DIR, "results.txt")),
        load_file(os.path.join(OUTPUT_DIR, "references.txt"))
    )

# ------------------ BUILD UI ------------------
with gr.Blocks() as app:
    gr.Markdown("# 📘 AI System: Automated Research Paper Review (Offline)")

    abstract_box = gr.Textbox(label="Abstract", lines=8)
    methods_box = gr.Textbox(label="Methods Comparison", lines=10)
    results_box = gr.Textbox(label="Results Synthesis", lines=10)
    refs_box = gr.Textbox(label="APA References", lines=8)

    load_button = gr.Button("Load Final Draft")
    load_button.click(
        fn=show_final_content,
        outputs=[abstract_box, methods_box, results_box, refs_box]
    )

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e2b96cf9345a20b0a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
app.launch()


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e2b96cf9345a20b0a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
